# Data x hora

In [ ]:
# Se marcan los casos donde a veces falla como
#"ERROR REPORTADO EN ALGUNOS CASOS"

In [ ]:
# ALg semiautomático

# AProvecha lo construido para operar de forma semiautomática

# 1. Buscar resistencias y soportes (se puede actualizar por ej cada una semana)....esas serán las ordenes de compra durante toda la semana
# 2. Conectarse con mt5 y ejecutar y hacer el trallingstop

In [ ]:
# Ejecutar con revenAI
# version yf = 0.2.65

# V2: Cambios el 260111...rediseño de macroalgorítmo y continuación de desarrollo

In [ ]:
print('Conexion con mt5 en mt5.ipynb [en este mismo proyecto]')
print('Llevado a la práctica en 00_Conexion_mt5.ipynb')

In [ ]:
#sys.exit("Entre 'df_oc_all base inicial' y 'df_oc_all base final', hay status que pasan de Abierta a Activa...eso no puede ocurrir [ver id 11]")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import itertools
import tqdm
import random
import warnings
import sys
import pickle
import time

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt

import mplfinance as mpf
from typing import Union

In [ ]:
fuente = 'conjuntosN2' # carpeta donde se guardan los conjuntos de soportes y resistencias para cada valor y cada N

In [ ]:
carpeta_data = "../Data/"

In [ ]:
import MetaTrader5 as mt5
import pandas as pd

# Inicializar MT5 (Inicialmente, debe conextarse para obtener datos de precio)
mt5.initialize()

# Initialize MT5 connection
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()
    sys.exit()

In [ ]:
valores = ['BTCUSD', 'ETHUSD', 'TSLA', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'NFLX']

# Funciones

In [ ]:
def pickle_act(file_name, variable = None, mode = 'open', eliminar_si_problemas = False): #260120
    
    """
    Guarda o carga una variable utilizando la biblioteca pickle.

    Parameters:
        - file_path (str): La ruta al archivo pickle.
        - variable: La variable a guardar (si mode='save') o None (si mode='open').
        - mode (str): 'save' para guardar la variable, 'open' para cargar la variable.

    Returns:
        La variable cargada si mode='open' o None si mode='save'.
    """
    
    dic_mode = {'save': 'wb', 'open': 'rb'}
    #print('file_name', file_name)

    while True:
        try:
            with open(f'{file_name}.pkl', dic_mode[mode]) as file:
                if mode == 'save':
                    pickle.dump(variable, file)
                    return None
                else:
                    #print('file en funciones transversales', f'{file_name}.pkl')
                    if eliminar_si_problemas:
                        try:
                            variable = pickle.load(file)
                        except:
                            os.remove(f'{file_name}.pkl')
                            return pd.DataFrame()
                    else:
                        try:
                            variable = pickle.load(file)
                        except Exception as e:
                            print('Problema en pickle_act 1:', e)
                            try:
                                with open(f'{file_name}.pkl', 'wb') as f:
                                    pickle.dump(df, f)
                            except Exception as e:
                                print('Problema en pickle_act 2:', e)
                                print(f"Intentar ejecutar with open(f'{file_name}.pkl', 'wb') as f: pickle.dump(df, f)")
                                sys.exit('A')
                    return variable
        except Exception as e:
            print(f'Problema en pickle_act 3: {e}')
            sys.exit()
            #time.sleep(10)
    
    return None

# Funciones MT5

In [ ]:
def obtener_conjuntos_actuales(valor, dic_seguimiento):
    lista_OA, lista_OE = [], []
    # posiciones abiertas actualmente (de ese valor)
    #print('A01')
    actual_OA = mt5.positions_get(symbol = valor)
    #print('ERROR REPORTADO EN ALGUNOS CASOS')
    for n in actual_OA:
        lista_OA.append(n.price_open)
    #print('A03')
    # posiciones pendientes actualmente (de ese valor)
    actual_OE = mt5.orders_get(symbol = valor)
    for n in actual_OE:
        lista_OE.append(n.price_open)
    #print('A04')
    # Actual OA
    if valor in dic_seguimiento:
        for orden in dic_seguimiento[valor]:
            if orden not in actual_OA:
                print('La orden ya no está abierta, se elimina de seguimiento:', orden)
                print('ACTUAL OA', actual_OA)
                print('Dic seguimiento antes de eliminar:', dic_seguimiento[valor])
                dic_seguimiento[valor].remove(orden)
    #print('A05')
    return lista_OA, lista_OE, actual_OA, actual_OE, dic_seguimiento

In [ ]:
def obtener_precio_actual(valor, modo = 'B'):
    # modo = A: up (ask), B: down (bid) [en compra]
    #print(mt5.symbol_info_tick(valor).bid, mt5.symbol_info_tick(valor).ask)
    if modo == 'B':
        return mt5.symbol_info_tick(valor).bid
    else:
        return mt5.symbol_info_tick(valor).ask

In [ ]:
def limpiar_ordenes_pendientes_no_validas(valor, actual_OE, lista_N):
    accion = False
    for i in range(len(actual_OE)):
        orden = actual_OE[i]
        precio_OE = orden.price_open
        precio_OE = round(precio_OE, 2)  # Redondear a 2 decimales para evitar problemas de precisión
        if precio_OE not in lista_N:
            accion = True
            
            ticket = orden.ticket
            request = {
                "action": mt5.TRADE_ACTION_REMOVE,
                "order": ticket,
                "symbol": valor,
                "type": orden.type,
                "position": orden.position_id,
                "comment": "Eliminacion de orden", # por estructura, debe terminar con coma (',)
            }
            result = mt5.order_send(request)
            if result is None:
                print(f"order_send failed, error code={mt5.last_error()}")
            elif result.retcode != mt5.TRADE_RETCODE_DONE:
                if result.retcode not in [10018]: # 10018: Mercado cerrado
                    print(f"Failed to remove order {ticket}, retcode={result.retcode}")
            else:
                print(f'Eliminando orden pendiente en {valor} a precio {precio_OE}')
                print(f"Order {ticket} removed successfully")
    return accion

In [ ]:
def generate_request_buy_limit(valor, order_type = '', sl = 0, volumen = 0, precio = 0): # 260122

    symbol = valor
    request = {
        "action": mt5.TRADE_ACTION_PENDING,
        "symbol": symbol,
        "volume": volumen,
        "type": order_type,
        "price": precio,
        "sl": float(sl),
        "deviation": 10,
        "magic": 123456,
        "comment": "Orden BUY desde Python",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_FOK,
    }
    
    if sl == 0:
        del request['sl']
    
    return request

In [ ]:
def ejecutar_orden(request, symbol, volumen, precio): # 260122
    # Se pone la orden de compra
    if 'sl' in request.keys(): #250902
        request['sl'] = float(round(request['sl'], 0))
        
    if volumen == 0:
        print('No se ejecuta por volumen = 0')
        return None
    # send a trading request
    result = mt5.order_send(request)
    #p
    try:
        if (result.retcode != mt5.TRADE_RETCODE_DONE):
            #print("2. order_send failed, retcode={}".format(result.retcode))
            
            # request the result as a dictionary and display it element by element
            result_dict = result._asdict()
            #print('result_dict', result_dict['comment'])
            
            if result_dict['comment'] == 'Market closed':
                #print('Market closed ok')
                return None # No se puede hacer nada por ahora, porque el mercado de ese valor está cerrado
            if result.retcode in [10006, 10044, 10018, 10031]: # 10044: Ejecución no permitida, 10018: Mercado cerrado
                #print('Ejecución no permitida (al menos por ahora)', result.retcode)
                return None

            for field in result_dict.keys():
                print('RDict', result_dict['comment'])
                print('Res', (result.retcode, mt5.TRADE_RETCODE_DONE), result)
                print("   {}={}".format(field, result_dict[field]))
                # if this is a trading request structure, display it element by element as well
                if field == "request":
                    traderequest_dict = result_dict[field]._asdict()
                    for tradereq_filed in traderequest_dict:
                        print("       traderequest: {}={}".format(tradereq_filed, traderequest_dict[tradereq_filed]))
                else:
                    print('Warning: field no es request', field)
                    
            print("shutdown() and quit")
            mt5.shutdown()
            quit()
        else:
            # check the execution result
            print("1. order_send(): by {} {} lots at {} with deviation={} points".format(symbol, volumen, precio, 10));
            print('result', result)
    except Exception as e:
        print(e)
        print(result)
    
    return None

In [ ]:
def crear_ordenes_espera(lista_OA, lista_OE, lista_N, valor, L, a, lotajes):
    #print('D1')
    P0 = obtener_precio_actual(valor, modo = 'B') # Precio actual de mercado (bid)
    #print('D2')
    lista_OAE = lista_OA + lista_OE # Contiene ordenes abiertas / activas (OA) y ordenes pendientes / en espera (OE)
    accion = False
    for Pi in lista_N:
        if Pi in lista_OAE:
            continue # ya existe, no agregar de nuevo
        if (P0 - Pi) * L >= a: # Si se cumple la distancia mínima
            accion = True
            # Activar orden de compra
            order_type = mt5.ORDER_TYPE_BUY_LIMIT
            volumen = lotajes[valor]
            #print('D25')
            request = generate_request_buy_limit(valor, order_type = order_type, volumen = volumen, precio = Pi) # Por ahora, sin SL de largo plazo, que es un argumento opcional de la función
            #print('volumen', volumen)
            #print('D26')
            ejecutar_orden(request, symbol = valor, volumen = volumen, precio = Pi)
            #print('D27')
    #print('D3')
    return accion

In [ ]:
def cambiar_SL(orden, valor, sl):
    #print('newSL', sl)
    ticket = orden.ticket
    request = {
        "action": mt5.TRADE_ACTION_SLTP,
        "symbol": valor,
        "position": ticket,
        "sl": float(round(sl, 2)),
        "deviation": 10,
        "comment": "SL0"
    }
    result = mt5.order_send(request)
    if result is None:
        print(f"order_send failed, error code={mt5.last_error()}")
    elif result.retcode != mt5.TRADE_RETCODE_DONE:
        print(f"Failed to modify SL for order {ticket} sl = {sl}, retcode={result.retcode}, code={mt5.last_error()}")
        print('request')
        print(request)
    else:
        print(f"SL for order {ticket} modified successfully to {sl}")
    return None

In [ ]:
# C: Trailing Stop
def trailing_stop(actual_OA, valor, L, a, b, lotajes, dic_seguimiento, minutos = 0):
    accion = False
    if len(actual_OA) == 0: # No hay ordenes abiertas
        time.sleep(minutos * 60)
        return None
    P0 = obtener_precio_actual(valor, modo = 'B')
    for orden in actual_OA:
        sl = orden.sl
        Pi = orden.price_open
        #print('Pi, sl, (P0 - Pi) * L, a', Pi, sl, (P0 - Pi) * L, a)
        cambios_orden = False
        if sl == 0:
            if (P0 - Pi) * L >= a:
                accion = True
                sl = P0 - b / L # Primer SL ganador (L normaliza b: b = 2 usd significan b /L = 200 usd en el precio real)
                #print('cambiar_SL')
                cambiar_SL(orden, valor, sl)
                # Además, se vuelve a generar la orden de compra
                order_type = mt5.ORDER_TYPE_BUY_LIMIT
                volumen = lotajes[valor]
                request = generate_request_buy_limit(valor, order_type = order_type, volumen = volumen, precio = Pi) # Por ahora, sin SL de largo plazo, que es un argumento opcional de la función
                #print('volumen', volumen)
                ejecutar_orden(request, symbol = valor, volumen = volumen, precio = Pi)
                cambios_orden = True
                
        else:
            accion = True
            new_SL = P0 - b / L # (L normaliza b: b = 2 usd significan b /L = 200 usd en el precio real)
            
            if new_SL > sl:
                print('P0-L-newSL-oldSL', valor, P0, L, new_SL, sl)
                print('cambiar SL')
                cambiar_SL(orden, valor, new_SL)
                cambios_orden = True
        
        if cambios_orden:
            if valor not in dic_seguimiento:
                dic_seguimiento[valor] = []
            if orden not in dic_seguimiento[valor]:
                dic_seguimiento[valor].append(orden)
                
    return accion, dic_seguimiento

In [ ]:
def leer_lista_N(valor, N, fuente = fuente):
    #print(f'Lectura conjuntosN/{valor}_{N}_beta')
    for i in range(10):
        if f'{valor}_{N}_beta.pkl' in os.listdir(f'{fuente}/'):
            lista_N = list(pickle_act(f'{fuente}/{valor}_{N}_beta'))
            lista_N = [round(n, 2) for n in lista_N]
            break
        time.sleep(2) # sleep de 2 segundos

    return lista_N

# Algorítmo

In [ ]:
from Transversal import n_sizes_ejecucion as n_sizes

In [ ]:
valores = ['BTCUSD', 'ETHUSD', 'GOOGL', 'TSLA', 'NVDA', 'AMZN'] # 'ETHUSD', 'GOOGL', 'TSLA'
a, b = 6, 2 # a es el minimo de ganancia en usd para poner SL ganador por primera vez. b es la distancia que debe mantener el SL con respecto al precio

In [ ]:
prueba_trailing_stop = False # False habilita la creación de nuevas ordenes de compra en espera (buy limit). True solo prueba el trailing stop en ordenes ya existentes

In [ ]:
# Mínimo y tambien cambio discreto (debe ser un ponderador de)
lotajes = {'BTCUSD': 0.01,
           'ETHUSD': 0.1,
           'ISRG': 0.01,
           'TSLA': 0.01,
           'GOOGL': 0.01,
           'NVDA': 0.01,
           'AMZN': 0.01}

units = {'BTCUSD': 1, # unidades del activo, en cada lote
           'ETHUSD': 1,
           'ISRG': 100, # 100 acciones por lote
           'TSLA': 100,
           'GOOGL': 100,
           'NVDA': 100,
              'AMZN': 100}

In [ ]:
if prueba_trailing_stop:
    valores = list(lotajes.keys()) 

valores

In [ ]:
# Ejemplo demo
#lista_N: 100 valores random uniforme entre 80000 y 90000.. pueden incluir dos decimales
#lista_N = [round(random.uniform(80000, 90000), 2) for _ in range(100)]
#lista_N.sort()
#lista_N[:3]
#a, b = 1, 0.2

In [ ]:
for valor in valores:
    info = mt5.symbol_info(valor)
    if info is None:
        print("El símbolo no existe en este bróker:", valor)
    else:
        print("Encontrado:", info.name, "visible:", info.visible)

    # Si existe pero no está visible, selecciónalo:
    if info and not info.visible:
        ok = mt5.symbol_select(valor, True)
        print("symbol_select:", ok, mt5.last_error())

In [ ]:

def is_open_by_tick(symbol, max_age=120):
    if not mt5.initialize():
        return False
    mt5.symbol_select(symbol, True)
    tick = mt5.symbol_info_tick(symbol)  # [2](https://www.mql5.com/en/docs/python_metatrader5/mt5symbolinfotick_py)
    mt5.shutdown()
    if tick is None:
        return False
    return (time.time() - tick.time) <= max_age


In [ ]:
# ANCLA 987
def informacion(valores, n_head = 10):
    df_N_all = pd.DataFrame()
    for valor in valores:
        #print(valor)
        P0 = obtener_precio_actual(valor, modo = 'B')
        
        N = n_sizes[valor]
        lista_N = leer_lista_N(valor, N, fuente = fuente)
        lista_N.sort()
        df_lista_N = pd.DataFrame(lista_N, columns = ['Precio'])
        
        L = lotajes[valor] * units[valor] # lotaje
        #print('Precio actual', P0)
        
        df_lista_N['DISTANCIA'] = (P0 - df_lista_N['Precio']) * L
        df_lista_N = df_lista_N[df_lista_N['DISTANCIA'] >= 0]
        df_lista_N['VALOR'] = valor
        df_N_all = pd.concat([df_N_all, df_lista_N], axis = 0)


    df_N_all = df_N_all.sort_values('DISTANCIA', ascending = True).reset_index(drop = True)
    df_N_all = df_N_all[['VALOR', 'Precio', 'DISTANCIA']]
    df_lista_N_por_declarar = df_N_all[df_N_all['DISTANCIA'] < a]
    df_lista_N_declarados = df_N_all[df_N_all['DISTANCIA'] >= a]

    df_lista_N_por_declarar['DISTANCIA'] = a - df_lista_N_por_declarar['DISTANCIA']
    df_lista_N_por_declarar = df_lista_N_por_declarar.sort_values('DISTANCIA', ascending = True).reset_index(drop = True)
    display('df_lista_N_por_declarar', df_lista_N_por_declarar.head(n_head))
    display('df_lista_N_declarados', df_lista_N_declarados.head(n_head))
    
    return None


In [ ]:
valores

In [ ]:
os.listdir(fuente)

In [ ]:
for valor in valores:
    N = n_sizes[valor]
    lista_N = leer_lista_N(valor, N, fuente = fuente)
    lista_N.sort()
    df_lista_N = pd.DataFrame(lista_N, columns = ['Precio'])
    display(valor, df_lista_N)

In [ ]:
mt5.terminal_info()
account = mt5.account_info()
account.trade_expert

In [ ]:
print('Inicio')
t0 = time.time()
ts = 0.5
dic_seguimiento = {}
i = 0
accion0, accion1, accion2 = False, False, False
while True:
    time_sleep = True
    for valor in valores:
        if i == 0:
            print('VALOR', valor)
        L = lotajes[valor] * units[valor] # lotaje
        if not prueba_trailing_stop:
            N = n_sizes[valor] # cantidad de soportes y resistencias a considerar
            lista_N = leer_lista_N(valor, N, fuente)
        
        #print(lista_N)
        # conjuntos OA y OE(ancla 1)
        lista_OA, lista_OE, actual_OA, actual_OE, dic_seguimiento = obtener_conjuntos_actuales(valor, dic_seguimiento) # listas solo incluyen los precios
        #if i == 0:
        #    print('VALOR', valor)
        #    print(actual_OA)
        #    print(actual_OE)
        # Precio actual del valor
        #P0 = obtener_precio_actual(valor, modo = 'D')
        # A: Para cada orden i en OE, si no está en listaN, eliminar de OE
        #print('B1')
        if (not prueba_trailing_stop) and (i % int(5 / ts) == 0): # Solo en la primera iteración, para evitar eliminar ordenes en espera creadas en esta misma ejecución
            accion0 = limpiar_ordenes_pendientes_no_validas(valor, actual_OE, lista_N)
        #sys.exit('Prueba')
        # B: Crear Ordenes en espera
       #print('B2')
        if not prueba_trailing_stop:
            accion1 = crear_ordenes_espera(lista_OA, lista_OE, lista_N, valor, L, a, lotajes)
        #print('B3')
        # C: Trailing Stop en posiciones abiertas
        accion2 = trailing_stop(actual_OA, valor, L, a, b, lotajes, dic_seguimiento)
        #print('B4')
        for c in dic_seguimiento: # Para cualquier valor con ordenes en seguimiento
            if len(dic_seguimiento[c]) > 0: # Si existe alguna orden abierta con SL, entonces no hay time sleep
                print(f'Seguimiento de {c}: {len(dic_seguimiento[c])} ordenes')
                print('dic_seguimiento 260416', dic_seguimiento)
                time_sleep = False # Se desactiva, es decir, si ocurre algo, se vuelve a ejecutar sin esperar
        #print('B5')
    
    if i % int(5000 / ts) == 0:
        print(f'Iteracion {i / 1000} M completada')
    
    if i % int(5000 / ts) == 0:
        print('Tiempo transcurrido (minutos):', round((time.time() - t0) / 60, 2))
        informacion(valores)
    i += 1
    if time_sleep:
        time.sleep(ts) 

    #sys.exit()


Iteracion 40.0 M completada
Tiempo transcurrido (minutos): 437.28


'df_lista_N_por_declarar'

,VALOR,Precio,DISTANCIA
0,NVDA,215.99,0.330
1,GOOGL,357.30,0.700
2,AMZN,252.21,0.810
3,TSLA,418.33,1.890
4,AMZN,253.60,2.200
5,ETHUSD,1861.97,2.783
6,NVDA,218.49,2.830
7,AMZN,254.98,3.580
8,GOOGL,360.97,4.370
9,ETHUSD,1883.62,4.948


'df_lista_N_declarados'

,VALOR,Precio,DISTANCIA
14,GOOGL,356.53,6.0700
15,BTCUSD,66732.93,6.3069
16,AMZN,250.84,6.5600
17,ETHUSD,1828.03,6.6110
18,AMZN,249.53,7.8700
19,TSLA,414.46,7.9800
20,NVDA,213.20,8.4600
21,GOOGL,353.88,8.7200
22,AMZN,248.27,9.1300
23,GOOGL,352.79,9.8100


...

Iteracion 60.0 M completada
Tiempo transcurrido (minutos): 616.44


'df_lista_N_por_declarar'

,VALOR,Precio,DISTANCIA
0,NVDA,215.99,0.3300
1,GOOGL,357.30,0.7000
2,AMZN,252.21,0.8100
3,TSLA,418.33,1.8900
4,AMZN,253.60,2.2000
5,ETHUSD,1828.03,2.3580
6,NVDA,218.49,2.8300
7,AMZN,254.98,3.5800
8,GOOGL,360.97,4.3700
9,BTCUSD,66732.93,4.9077


'df_lista_N_declarados'

,VALOR,Precio,DISTANCIA
14,GOOGL,356.53,6.0700
15,AMZN,250.84,6.5600
16,ETHUSD,1795.79,6.8660
17,AMZN,249.53,7.8700
18,TSLA,414.46,7.9800
19,BTCUSD,66018.64,8.2352
20,NVDA,213.20,8.4600
21,GOOGL,353.88,8.7200
22,AMZN,248.27,9.1300
23,GOOGL,352.79,9.8100


In [ ]:
sys.exit('Otros')

In [ ]:
mt5.initialize()

Prueba de conexión

In [ ]:

import MetaTrader5 as mt5
print("MT5 package version:", mt5.__version__)
assert mt5.initialize(), mt5.last_error()
print("Terminal info:", mt5.terminal_info())
print("MT5 version:", mt5.version())


In [ ]:

symbol = "BTCUSD"  # <- prueba luego variaciones: BTCUSD.r, BTCUSDm, BTCUSDT, etc.

info = mt5.symbol_info(symbol)
if info is None:
    print("El símbolo no existe en este bróker:", symbol)
else:
    print("Encontrado:", info.name, "visible:", info.visible)

# Si existe pero no está visible, selecciónalo:
if info and not info.visible:
    ok = mt5.symbol_select(symbol, True)
    print("symbol_select:", ok, mt5.last_error())


In [ ]:

tick = mt5.symbol_info_tick(symbol)
if tick is None:
    print("Sin tick para", symbol, "->", mt5.last_error())
else:
    print("Bid:", tick.bid, "Ask:", tick.ask, "Time:", tick.time)


In [ ]:
sys.exit('Otros abajo')

In [ ]:
a, b = 0.1, 0.05

In [ ]:
# Colocar SL ganador por primera vez
nuevo_sl = P0 - b
ticket = orden.ticket
request = {
    "action": mt5.TRADE_ACTION_SLTP,
    "symbol": valor,
    "position": ticket,
    "sl": float(round(nuevo_sl, 0)),
    "deviation": 10,
    "comment": "SL0"
}
result = mt5.order_send(request)
if result is None:
    print(f"order_send failed, error code={mt5.last_error()}")
elif result.retcode != mt5.TRADE_RETCODE_DONE:
    print(f"Failed to modify SL for order {ticket}, retcode={result.retcode}")
else:
    print(f"SL for order {ticket} modified successfully to {nuevo_sl}")

In [ ]:
sys.exit('Otros')

In [ ]:
# ancla 1

# Conecta

In [ ]:
# Establish connection to MT5
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()

# Get all pending orders
# Use mt5.orders_get() to retrieve all orders.
# Filter for pending orders based on their type (e.g., mt5.ORDER_TYPE_BUY_LIMIT, mt5.ORDER_TYPE_SELL_STOP, etc.)
# A more robust approach would be to check if the order is a pending order.
# obtener ordenes abiertas actualmente
actual_positions = mt5.positions_get(symbol = valor)
actual_positions

In [ ]:
actual_positions[0].price_open

In [ ]:
actual_orders[0]

In [ ]:
actual_orders[0].price_open

In [ ]:
lista = [1, 2, 3, 4, 5]

# ordena la lista de forma random
random.shuffle(lista)
lista